In [1]:
import pandas as pd
import numpy as np
import torch
import os
import kagglehub
from collections import defaultdict



In [ ]:
path = kagglehub.dataset_download("mohamedbakhet/amazon-books-reviews")

# Загрузка Books_rating
df = pd.read_csv(os.path.join(path, 'Books_rating.csv'))

In [ ]:
df = df.rename(columns={
    'User_id': 'user_id',
    'Id': 'book_id',
    'Title': 'title',
    'review/score': 'rating',
    'review/time': 'date_read'
})
df['date_read'] = pd.to_datetime(df['date_read'], unit='s', errors='coerce')

# Загрузка books_data
books = pd.read_csv(os.path.join(path, 'books_data.csv'))
books = books.rename(columns={
    'Title': 'title'
})

In [ ]:
# Объединяем по названию книги
df = df.merge(books[['title', 'authors', 'categories']], on='title', how='left')
print(f"Загружено {len(df):,} взаимодействий")
print(f"Пользователей: {df['user_id'].nunique():,}")
print(f"Книг: {df['book_id'].nunique():,}")
print(f"Колонки: {df.columns.tolist()}")

In [ ]:
print(f"Уникальных авторов ДО базовой очистки: {df['authors'].nunique()}")
print(f"Уникальных категорий ДО базовой очистки: {df['categories'].nunique()}")

# убираем квадратные скобки и кавычки, точки и запятые и пробелы
df['authors'] = df['authors'].apply(
    lambda x: x.strip("[]'").replace("'", "") if isinstance(x, str) else 'unknown'
)
df['categories'] = df['categories'].apply(
    lambda x: x.strip("[]'").replace("'", "") if isinstance(x, str) else 'unknown'
)

df['authors'] = df['authors'].str.strip().str.lower().str.replace('.', '', regex=False)
df['authors'] = df['authors'].str.replace(r'\s+', ' ', regex=True)

df['categories'] = df['categories'].str.strip().str.lower().str.replace('.', '', regex=False)
df['categories'] = df['categories'].str.replace(r'\s+', ' ', regex=True)

print(f"Уникальных авторов после базовой очистки: {df['authors'].nunique()}")
print(f"Уникальных категорий после базовой очистки: {df['categories'].nunique()}")

In [ ]:
# Поиск неявных дубликатов по фамилии и инициалам

author_counts = df['authors'].value_counts()
author_list = [a for a in author_counts.index if a != 'unknown']

# разбиваем на слова и классифицируем
full_names = []
initial_names = []

for author in author_list:
    words = author.split()
    if len(words) < 2:
        continue
    if all(len(w) == 1 for w in words[:-1]):
        initial_names.append((author, words))
    else:
        full_names.append((author, words))

# cравниваем только полные имена с инициалами
for full_author, full_words in full_names:
    full_lastname = full_words[-1]
    full_initials = ''.join(w[0] for w in full_words[:-1])

    for init_author, init_words in initial_names:
        init_lastname = init_words[-1]

        if full_lastname != init_lastname:
            continue

        init_initials = ''.join(init_words[:-1])

        if full_initials.startswith(init_initials):
            cnt1 = author_counts[full_author]
            cnt2 = author_counts[init_author]
            if cnt1 + cnt2 > 500:
                print(f"'{full_author}' ({cnt1:,}) ↔ '{init_author}' ({cnt2:,})")
                print()

In [ ]:
# исправляем найденные дубликаты
author_mapping = {
    'john ronald reuel tolkien': 'j r r tolkien',
    'john ronald reuel tolkien, christopher tolkien': 'j r r tolkien',
    'r l stevenson': 'robert louis stevenson',
    'robert louis stevenson, fanny van de grift stevenson': 'robert louis stevenson',
    'h melville': 'herman melville',
    'a smith': 'adam smith',
    'v woolf': 'virginia woolf',
    'd gabaldon': 'diana gabaldon',
    'b anderson': 'brian herbert, kevin j anderson',
    'clive staples lewis': 'c s lewis',
    'clive s lewis': 'c s lewis',
    'carolyn sherwin bailey, clara m lewis': 'c s lewis',
    'm clark': 'mary higgins clark',
    'm h clark': 'mary higgins clark',
    'l m montgomery': 'lucy maud montgomery',
    'd h lawrence': 'david herbert lawrence',
    'j swift': 'jonathan swift',
    'g k chesterton': 'gilbert keith chesterton',
    't h white': 'terence hanbury white',
    'e b white': 'elwyn brooks white',
    'elwyn brooks white, katharine sergeant angell white': 'elwyn brooks white',
    'j d watson': 'james d watson',
    'cecil scott forester': 'c s forester',
    'h jacobs': 'harriet jacobs',
    'm phillips': 'michael phillips',
    'a a milne': 'alan alexander milne',
    'p g wodehouse': 'pelham grenville wodehouse',
    'p wodehouse': 'pelham grenville wodehouse',
    'h a rey, margret rey': 'h a rey',
    'jerome david salinger': 'j d salinger',
    'e m forster': 'edward morgan forster',
    'edward m forster': 'edward morgan forster',
    'h g wells': 'herbert george wells',
    'herbert g wells': 'herbert george wells',
    'r l (written by kathryn lance) stine': 'r l stine',
    'a j frost, richard russell': 'a j russell',
    'r h dana': 'richard henry dana',
}

df['authors'] = df['authors'].replace(author_mapping)
print(f"Уникальных авторов после маппинга: {df['authors'].nunique()}")

# заполняем пропуски
df['authors'] = df['authors'].fillna('unknown')
df['categories'] = df['categories'].fillna('unknown')

In [ ]:
def preprocess_data(df, min_user_books=5, max_seq_len=30,
                    val_ratio=0.15, test_ratio=0.15):

    # фильтрация
    filtered_rows = []
    for _, row in df.iterrows():
        filtered_rows.append({
            'uid': row['user_id'],
            'item_ids': row['book_id'],
            'author_ids': row['authors'],
            'category_ids': row['categories'],
            'date_read': row['date_read']
        })
    df = pd.DataFrame(filtered_rows)

     # Группировка по пользователям
    user_data = df.groupby('uid').agg({
        'item_ids': list,
        'author_ids': list,
        'category_ids': list,
        'date_read': list
    }).reset_index()

    user_data = user_data[user_data['item_ids'].apply(len) >= min_user_books]
    print(f"\nПосле фильтрации (>= {min_user_books} книг): {len(user_data)} пользователей")
